# **Combine real and synthetic data for Data-Efficient Agricultural Computer Vision**

This notebook investigates the use of **synthetic data to complement limited real agricultural images** for data-efficient computer vision.

To evaluate performance stability, we randomly select **10 real images** for training and repeat the experiment **5 times** with different random selections. Results are reported as **mean ± standard deviation**, providing an estimate of model performance and robustness across different training subsets.

Any problems, contact:
wenzhi.liao@flandersmake.be

## for carrot and apple datasets

In [ ]:
import os
import shutil
import random
import csv
import numpy as np
from pathlib import Path
from ultralytics import YOLO


# ============================================================
# CONFIGURATION
# ============================================================

# ------------------------------------------------------------
# Original REAL training dataset
# ------------------------------------------------------------


REAL_IMAGES_DIR = Path("apples/Real/images/train")
REAL_LABELS_DIR = Path("apples/Real/labels/train")

# ------------------------------------------------------------
# SYNTHETIC dataset
# ------------------------------------------------------------

SYNTHETIC_IMAGES_DIR = Path("apples/CAD2Render/images")
SYNTHETIC_LABELS_DIR = Path("apples/CAD2Render/labels")


# ------------------------------------------------------------
# VALIDATION dataset
# Used during training/model development
# ------------------------------------------------------------

VAL_IMAGES_DIR = Path("apples/Real/images/val")
VAL_LABELS_DIR = Path("apples/Real/labels/val")


# ------------------------------------------------------------
# TEST dataset
# Used ONLY for final evaluation
# ------------------------------------------------------------

TEST_IMAGES_DIR = Path("apples/Real/images/test")
TEST_LABELS_DIR = Path("apples/Real/labels/test")


# ------------------------------------------------------------
# YOLO model
# ------------------------------------------------------------

MODEL_PATH = "yolo11n-seg.pt"


# ------------------------------------------------------------
# Experiment settings
# ------------------------------------------------------------

# Number of real images randomly selected
# for each experiment
N_REAL_IMAGES = 10

# Number of independent experiments
N_EXPERIMENTS = 5

# Training
EPOCHS = 100
IMAGE_SIZE = 640
BATCH_SIZE = 16
DEVICE = 0       # change to "cpu" if needed


# ------------------------------------------------------------
# Output
# ------------------------------------------------------------

WORK_DIR = Path("random_experiments_apple")


# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

BASE_SEED = 42


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def get_image_files(images_dir):
    """
    Return all supported image files in a directory.
    """

    extensions = {
        ".jpg",
        ".jpeg",
        ".png",
        ".bmp",
        ".tif",
        ".tiff"
    }

    return sorted([
        p
        for p in images_dir.iterdir()
        if p.is_file()
        and p.suffix.lower() in extensions
    ])


def copy_image_and_label(
    image_path,
    source_label_dir,
    destination_image_dir,
    destination_label_dir
):
    """
    Copy an image and its corresponding YOLO segmentation label.
    """

    destination_image_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    destination_label_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    # --------------------------------------------------------
    # Copy image
    # --------------------------------------------------------

    shutil.copy2(
        image_path,
        destination_image_dir / image_path.name
    )

    # --------------------------------------------------------
    # Find corresponding annotation
    # --------------------------------------------------------

    label_path = (
        source_label_dir /
        f"{image_path.stem}.txt"
    )

    if not label_path.exists():

        raise FileNotFoundError(
            f"\nAnnotation not found!\n"
            f"Image: {image_path}\n"
            f"Expected annotation: {label_path}"
        )

    shutil.copy2(
        label_path,
        destination_label_dir / label_path.name
    )


def copy_directory_contents(
    source_images,
    source_labels,
    destination_images,
    destination_labels
):
    """
    Copy all images + labels from one dataset
    into another dataset.
    """

    destination_images.mkdir(
        parents=True,
        exist_ok=True
    )

    destination_labels.mkdir(
        parents=True,
        exist_ok=True
    )

    images = get_image_files(source_images)

    for image_path in images:

        copy_image_and_label(
            image_path,
            source_labels,
            destination_images,
            destination_labels
        )

    return len(images)


def create_experiment_dataset(
    experiment_dir,
    selected_real_images
):
    """
    Creates:

        experiment_X/
            images/
                train/
                val/
                test/
            labels/
                train/
                val/
                test/

    Training:
        ALL synthetic images
        +
        selected real images

    Validation:
        fixed validation dataset

    Test:
        fixed independent test dataset

    The test dataset is copied into each experiment directory
    but is NEVER used during training.
    """

    train_images = (
        experiment_dir /
        "images" /
        "train"
    )

    train_labels = (
        experiment_dir /
        "labels" /
        "train"
    )

    val_images = (
        experiment_dir /
        "images" /
        "val"
    )

    val_labels = (
        experiment_dir /
        "labels" /
        "val"
    )

    test_images = (
        experiment_dir /
        "images" /
        "test"
    )

    test_labels = (
        experiment_dir /
        "labels" /
        "test"
    )


    # ========================================================
    # 1. Copy synthetic training dataset
    # ========================================================

    n_synthetic = copy_directory_contents(
        SYNTHETIC_IMAGES_DIR,
        SYNTHETIC_LABELS_DIR,
        train_images,
        train_labels
    )


    # ========================================================
    # 2. Copy selected real images into training dataset
    # ========================================================

    for image_path in selected_real_images:

        copy_image_and_label(
            image_path,
            REAL_LABELS_DIR,
            train_images,
            train_labels
        )


    # ========================================================
    # 3. Copy validation dataset
    # ========================================================

    n_val = copy_directory_contents(
        VAL_IMAGES_DIR,
        VAL_LABELS_DIR,
        val_images,
        val_labels
    )


    # ========================================================
    # 4. Copy independent test dataset
    #
    # IMPORTANT:
    # These images are NOT included in training or validation.
    # ========================================================

    n_test = copy_directory_contents(
        TEST_IMAGES_DIR,
        TEST_LABELS_DIR,
        test_images,
        test_labels
    )


    # ========================================================
    # 5. Create dataset YAML
    # ========================================================

    yaml_path = (
        experiment_dir /
        "dataset.yaml"
    )

    yaml_content = f"""
path: {experiment_dir.resolve()}

train: images/train
val: images/val
test: images/test

names:
  0: class0
"""

    yaml_path.write_text(
        yaml_content.strip()
    )


    # ========================================================
    # 6. Print dataset information
    # ========================================================

    print(
        f"Synthetic training images : "
        f"{n_synthetic}"
    )

    print(
        f"Selected real images      : "
        f"{len(selected_real_images)}"
    )

    print(
        f"Validation images         : "
        f"{n_val}"
    )

    print(
        f"Test images               : "
        f"{n_test}"
    )

    return yaml_path


def extract_metrics(metrics):
    """
    Extract requested metrics from Ultralytics
    segmentation validation results.
    """

    # --------------------------------------------------------
    # Bounding box metrics
    # --------------------------------------------------------

    bbox_map50 = float(
        metrics.box.map50
    )

    bbox_recall = float(
        metrics.box.r.mean()
    )


    # --------------------------------------------------------
    # Mask / segmentation metrics
    # --------------------------------------------------------

    mask_map50 = float(
        metrics.seg.map50
    )

    mask_recall = float(
        metrics.seg.r.mean()
    )


    return {
        "bbox_mAP50": bbox_map50,
        "mask_mAP50": mask_map50,
        "bbox_recall": bbox_recall,
        "mask_recall": mask_recall
    }


def calculate_mean_std(values):
    """
    Calculate mean and sample standard deviation.
    """

    values = np.asarray(
        values,
        dtype=float
    )

    mean = np.mean(values)

    std = np.std(
        values,
        ddof=1
    )

    return mean, std


# ============================================================
# MAIN
# ============================================================

def main():

    # ========================================================
    # CHECK DIRECTORIES
    # ========================================================

    required_dirs = [

        REAL_IMAGES_DIR,
        REAL_LABELS_DIR,

        SYNTHETIC_IMAGES_DIR,
        SYNTHETIC_LABELS_DIR,

        VAL_IMAGES_DIR,
        VAL_LABELS_DIR,

        TEST_IMAGES_DIR,
        TEST_LABELS_DIR
    ]


    for directory in required_dirs:

        if not directory.exists():

            raise FileNotFoundError(
                f"Directory does not exist:\n"
                f"{directory}"
            )


    # ========================================================
    # GET REAL TRAINING IMAGES
    # ========================================================

    real_images = get_image_files(
        REAL_IMAGES_DIR
    )


    # ========================================================
    # GET TEST IMAGES
    # ========================================================

    test_images = get_image_files(
        TEST_IMAGES_DIR
    )


    print("\n")
    print("=" * 70)
    print("YOLO11-SEG RANDOM SAMPLING EXPERIMENT")
    print("=" * 70)

    print(
        f"Real training images       : "
        f"{len(real_images)}"
    )

    print(
        f"Synthetic training images  : "
        f"{len(get_image_files(SYNTHETIC_IMAGES_DIR))}"
    )

    print(
        f"Validation images           : "
        f"{len(get_image_files(VAL_IMAGES_DIR))}"
    )

    print(
        f"Test images                 : "
        f"{len(test_images)}"
    )

    print(
        f"Real images/experiment      : "
        f"{N_REAL_IMAGES}"
    )

    print(
        f"Number of experiments       : "
        f"{N_EXPERIMENTS}"
    )

    print("=" * 70)


    if len(real_images) < N_REAL_IMAGES:

        raise ValueError(
            f"Only {len(real_images)} real images "
            f"available, but "
            f"{N_REAL_IMAGES} are required."
        )


    if len(test_images) == 0:

        raise ValueError(
            "No test images were found."
        )


    # ========================================================
    # CREATE OUTPUT DIRECTORY
    # ========================================================

    WORK_DIR.mkdir(
        parents=True,
        exist_ok=True
    )


    # ========================================================
    # STORE RESULTS
    # ========================================================

    results = []


    # ========================================================
    # RUN EXPERIMENTS
    # ========================================================

    for experiment_number in range(
        1,
        N_EXPERIMENTS + 1
    ):

        print("\n")
        print("=" * 70)

        print(
            f"EXPERIMENT "
            f"{experiment_number}/"
            f"{N_EXPERIMENTS}"
        )

        print("=" * 70)


        # ----------------------------------------------------
        # Unique random seed
        # ----------------------------------------------------

        seed = (
            BASE_SEED +
            experiment_number
        )

        rng = random.Random(seed)


        # ----------------------------------------------------
        # Randomly select real images
        # ----------------------------------------------------

        selected_real_images = rng.sample(
            real_images,
            N_REAL_IMAGES
        )


        print(
            "\nSelected real images:"
        )

        for image_path in selected_real_images:

            print(
                f"  {image_path.name}"
            )


        # ----------------------------------------------------
        # Experiment directory
        # ----------------------------------------------------

        experiment_dir = (
            WORK_DIR /
            f"experiment_{experiment_number}"
        )


        if experiment_dir.exists():

            shutil.rmtree(
                experiment_dir
            )


        # ----------------------------------------------------
        # Create combined dataset
        # ----------------------------------------------------

        yaml_path = create_experiment_dataset(
            experiment_dir,
            selected_real_images
        )


        # ----------------------------------------------------
        # Save selected image list
        # ----------------------------------------------------

        selected_file = (
            experiment_dir /
            "selected_real_images.txt"
        )


        with open(
            selected_file,
            "w"
        ) as f:

            for image_path in selected_real_images:

                f.write(
                    f"{image_path.name}\n"
                )


        # ====================================================
        # LOAD FRESH MODEL
        # ====================================================

        print("\nLoading fresh YOLO11n-seg model...")

        model = YOLO(
            MODEL_PATH
        )


        # ====================================================
        # TRAIN
        # ====================================================

        print("\nStarting training...")

        model.train(

            data=str(
                yaml_path
            ),

            epochs=EPOCHS,

            imgsz=IMAGE_SIZE,

            batch=BATCH_SIZE,

            device=DEVICE,

            project=str(
                WORK_DIR /
                "training_results"
            ),

            name=(
                f"experiment_"
                f"{experiment_number}"
            ),

            seed=seed,

            pretrained=True,

            verbose=True
        )


        # ====================================================
        # LOAD BEST WEIGHTS
        # ====================================================

        # best_weights = (
        #     WORK_DIR /
        #     "training_results" /
        #     f"experiment_{experiment_number}" /
        #     "weights" /
        #     "best.pt"
        # )

        best_weights = (
            Path("runs") /
            "segment" /
            WORK_DIR /
            "training_results" /
            f"experiment_{experiment_number}" /
            "weights" /
            "best.pt"
        )


        if not best_weights.exists():

            raise FileNotFoundError(
                f"Could not find best weights:\n"
                f"{best_weights}"
            )


        print(
            "\nBest model:"
        )

        print(
            best_weights
        )


        # ----------------------------------------------------
        # IMPORTANT:
        # Load the BEST checkpoint rather than using the
        # last training checkpoint.
        # ----------------------------------------------------

        best_model = YOLO(
            str(best_weights)
        )


        # ====================================================
        # FINAL EVALUATION ON TEST DATASET
        # ====================================================

        print("\n")
        print("=" * 70)

        print(
            "FINAL EVALUATION ON TEST DATASET"
        )

        print("=" * 70)


        test_metrics = best_model.val(

            data=str(
                yaml_path
            ),

            # IMPORTANT:
            # This is the validation API, but we explicitly
            # point it to the independent test dataset below.
            split="test",

            imgsz=IMAGE_SIZE,

            batch=BATCH_SIZE,

            device=DEVICE,

            verbose=True
        )


        # ====================================================
        # EXTRACT TEST METRICS
        # ====================================================

        experiment_metrics = extract_metrics(
            test_metrics
        )


        experiment_metrics[
            "experiment"
        ] = experiment_number

        experiment_metrics[
            "seed"
        ] = seed


        results.append(
            experiment_metrics
        )


        # ====================================================
        # PRINT EXPERIMENT RESULTS
        # ====================================================

        print("\n")
        print(
            f"TEST RESULTS - "
            f"EXPERIMENT "
            f"{experiment_number}"
        )

        print("-" * 50)

        print(
            f"bbox mAP50  : "
            f"{experiment_metrics['bbox_mAP50']:.4f}"
        )

        print(
            f"mask mAP50  : "
            f"{experiment_metrics['mask_mAP50']:.4f}"
        )

        print(
            f"bbox recall : "
            f"{experiment_metrics['bbox_recall']:.4f}"
        )

        print(
            f"mask recall : "
            f"{experiment_metrics['mask_recall']:.4f}"
        )


    # ========================================================
    # CALCULATE MEAN ± STD
    # ========================================================

    print("\n")
    print("=" * 70)
    print("FINAL TEST RESULTS: MEAN ± STD")
    print("=" * 70)


    metric_names = [

        "bbox_mAP50",
        "mask_mAP50",
        "bbox_recall",
        "mask_recall"
    ]


    summary = {}


    for metric in metric_names:

        values = [
            result[metric]
            for result in results
        ]


        mean_value, std_value = (
            calculate_mean_std(values)
        )


        summary[metric] = {

            "mean": mean_value,

            "std": std_value
        }


        print(
            f"{metric:15s}: "
            f"{mean_value:.4f} ± "
            f"{std_value:.4f}"
        )


    # ========================================================
    # SAVE INDIVIDUAL TEST RESULTS
    # ========================================================

    csv_path = (
        WORK_DIR /
        "test_results.csv"
    )


    with open(
        csv_path,
        "w",
        newline=""
    ) as f:

        writer = csv.DictWriter(

            f,

            fieldnames=[
                "experiment",
                "seed",
                "bbox_mAP50",
                "mask_mAP50",
                "bbox_recall",
                "mask_recall"
            ]
        )


        writer.writeheader()

        writer.writerows(
            results
        )


    # ========================================================
    # SAVE SUMMARY
    # ========================================================

    summary_path = (
        WORK_DIR /
        "test_summary.txt"
    )


    with open(
        summary_path,
        "w"
    ) as f:

        f.write(
            "YOLO11n-seg Test Dataset Results\n"
        )

        f.write(
            "=" * 60 +
            "\n\n"
        )


        f.write(
            f"Number of experiments: "
            f"{N_EXPERIMENTS}\n"
        )

        f.write(
            f"Real images per experiment: "
            f"{N_REAL_IMAGES}\n"
        )

        f.write(
            f"Test images: "
            f"{len(test_images)}\n\n"
        )


        for metric in metric_names:

            mean_value = (
                summary[metric]["mean"]
            )

            std_value = (
                summary[metric]["std"]
            )


            f.write(

                f"{metric}: "
                f"{mean_value:.4f} ± "
                f"{std_value:.4f}\n"
            )


    # ========================================================
    # FINAL MESSAGE
    # ========================================================

    print("\n")
    print("=" * 70)
    print("ALL EXPERIMENTS COMPLETED")
    print("=" * 70)

    print(
        "\nIndividual results:"
    )

    print(
        csv_path
    )

    print(
        "\nMean ± STD summary:"
    )

    print(
        summary_path
    )



In [ ]:
if __name__ == "__main__":

    main()

## repeat 5 times for only 10 real images

In [1]:
import shutil
import csv
import numpy as np
from pathlib import Path
from ultralytics import YOLO


# ============================================================
# CONFIGURATION
# ============================================================

# ------------------------------------------------------------
# Existing experiment folders
# ------------------------------------------------------------

WORK_DIR = Path(
    "random_experiments"
)

N_EXPERIMENTS = 5


# ------------------------------------------------------------
# Original REAL training dataset
# Used to locate the selected images and labels
# ------------------------------------------------------------

REAL_IMAGES_DIR = Path(
    "carrots/crack_all/images/Real"
)

REAL_LABELS_DIR = Path(
    "carrots/crack_all/labels/Real"
)


# ------------------------------------------------------------
# YOLO model
# ------------------------------------------------------------

MODEL_PATH = "yolo11n-seg.pt"


# ------------------------------------------------------------
# Training settings
# ------------------------------------------------------------

EPOCHS = 100
IMAGE_SIZE = 640
BATCH_SIZE = 16
DEVICE = 0


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def read_selected_images(txt_file):
    """
    Read image filenames from selected_real_images.txt.
    """

    if not txt_file.exists():
        raise FileNotFoundError(
            f"Selection file not found:\n{txt_file}"
        )

    with open(
        txt_file,
        "r",
        encoding="utf-8"
    ) as f:

        image_names = [
            line.strip()
            for line in f
            if line.strip()
        ]

    return image_names


# ============================================================

def prepare_train10(
    experiment_dir,
    selected_images
):
    """
    Create only:

        experiment_X/images/train10/
        experiment_X/labels/train10/

    using the 10 images listed in
    selected_real_images.txt.

    Existing train/val/test folders are NOT modified.
    """

    train10_images = (
        experiment_dir /
        "images" /
        "train10"
    )

    train10_labels = (
        experiment_dir /
        "labels" /
        "train10"
    )


    # --------------------------------------------------------
    # Remove previous train10 folder if it exists
    # --------------------------------------------------------

    if train10_images.exists():
        shutil.rmtree(train10_images)

    if train10_labels.exists():
        shutil.rmtree(train10_labels)


    train10_images.mkdir(
        parents=True,
        exist_ok=True
    )

    train10_labels.mkdir(
        parents=True,
        exist_ok=True
    )


    # --------------------------------------------------------
    # Copy only selected 10 images and labels
    # --------------------------------------------------------

    for image_name in selected_images:

        # ----------------------------------------------------
        # Source image
        # ----------------------------------------------------

        source_image = (
            REAL_IMAGES_DIR /
            image_name
        )

        if not source_image.exists():

            raise FileNotFoundError(
                f"\nSelected image does not exist:\n"
                f"{source_image}"
            )


        # ----------------------------------------------------
        # Source label
        # ----------------------------------------------------

        source_label = (
            REAL_LABELS_DIR /
            f"{source_image.stem}.txt"
        )

        if not source_label.exists():

            raise FileNotFoundError(
                f"\nCorresponding label does not exist:\n"
                f"{source_label}"
            )


        # ----------------------------------------------------
        # Copy image
        # ----------------------------------------------------

        shutil.copy2(
            source_image,
            train10_images /
            source_image.name
        )


        # ----------------------------------------------------
        # Copy label
        # ----------------------------------------------------

        shutil.copy2(
            source_label,
            train10_labels /
            source_label.name
        )


    print(
        f"Created train10 with "
        f"{len(selected_images)} images."
    )

    print(
        f"Images: {train10_images}"
    )

    print(
        f"Labels: {train10_labels}"
    )


    return train10_images, train10_labels


# ============================================================

def create_dataset_yaml(experiment_dir):
    """
    Create YOLO dataset YAML.

    IMPORTANT:
        train  -> existing train10
        val    -> existing val
        test   -> existing test

    No validation/test data are copied.
    """

    yaml_path = (
        experiment_dir /
        "dataset_train10.yaml"
    )


    yaml_content = f"""
path: {experiment_dir.resolve()}

train: images/train10
val: images/val
test: images/test

names:
  0: class0
"""


    yaml_path.write_text(
        yaml_content.strip(),
        encoding="utf-8"
    )


    return yaml_path


# ============================================================

def extract_metrics(metrics):
    """
    Extract requested YOLO segmentation metrics.
    """

    return {

        "bbox_mAP50":
            float(metrics.box.map50),

        "mask_mAP50":
            float(metrics.seg.map50),

        "bbox_recall":
            float(metrics.box.r.mean()),

        "mask_recall":
            float(metrics.seg.r.mean())
    }


# ============================================================

def calculate_mean_std(values):
    """
    Calculate mean and sample standard deviation.
    """

    values = np.asarray(
        values,
        dtype=float
    )

    mean = np.mean(values)

    std = np.std(
        values,
        ddof=1
    )

    return mean, std


# ============================================================
# MAIN
# ============================================================

def main():

    # ========================================================
    # CHECK ORIGINAL DATASET
    # ========================================================

    if not REAL_IMAGES_DIR.exists():

        raise FileNotFoundError(
            f"Real image directory does not exist:\n"
            f"{REAL_IMAGES_DIR}"
        )


    if not REAL_LABELS_DIR.exists():

        raise FileNotFoundError(
            f"Real label directory does not exist:\n"
            f"{REAL_LABELS_DIR}"
        )


    # ========================================================
    # CHECK EXPERIMENT FOLDERS
    # ========================================================

    experiment_dirs = []

    for i in range(
        1,
        N_EXPERIMENTS + 1
    ):

        experiment_dir = (
            WORK_DIR /
            f"experiment_{i}"
        )


        if not experiment_dir.exists():

            raise FileNotFoundError(
                f"Experiment directory does not exist:\n"
                f"{experiment_dir}"
            )


        # ----------------------------------------------------
        # Existing validation/test directories
        # ----------------------------------------------------

        required_dirs = [

            experiment_dir /
            "images" /
            "val",

            experiment_dir /
            "images" /
            "test",

            experiment_dir /
            "labels" /
            "val",

            experiment_dir /
            "labels" /
            "test"
        ]


        for directory in required_dirs:

            if not directory.exists():

                raise FileNotFoundError(
                    f"Required existing directory "
                    f"does not exist:\n"
                    f"{directory}"
                )


        # ----------------------------------------------------
        # selected_real_images.txt
        # ----------------------------------------------------

        selection_file = (
            experiment_dir /
            "selected_real_images.txt"
        )


        if not selection_file.exists():

            raise FileNotFoundError(
                f"Missing:\n"
                f"{selection_file}"
            )


        experiment_dirs.append(
            experiment_dir
        )


    # ========================================================
    # RESULTS
    # ========================================================

    results = []


    # ========================================================
    # RUN FIVE EXPERIMENTS
    # ========================================================

    for experiment_number, experiment_dir in enumerate(
        experiment_dirs,
        start=1
    ):

        print("\n")
        print("=" * 70)

        print(
            f"EXPERIMENT "
            f"{experiment_number}/{N_EXPERIMENTS}"
        )

        print("=" * 70)


        # ====================================================
        # READ SELECTED REAL IMAGES
        # ====================================================

        selection_file = (
            experiment_dir /
            "selected_real_images.txt"
        )


        selected_images = read_selected_images(
            selection_file
        )


        print(
            f"\nSelected real images: "
            f"{len(selected_images)}"
        )


        # ----------------------------------------------------
        # Verify exactly 10 images
        # ----------------------------------------------------

        if len(selected_images) != 10:

            raise ValueError(
                f"{selection_file} contains "
                f"{len(selected_images)} images. "
                f"Expected exactly 10."
            )


        for image_name in selected_images:

            print(
                f"  {image_name}"
            )


        # ====================================================
        # CREATE train10
        #
        # Existing val/test are NOT copied.
        # ====================================================

        prepare_train10(
            experiment_dir,
            selected_images
        )


        # ====================================================
        # CREATE DATASET YAML
        # ====================================================

        yaml_path = create_dataset_yaml(
            experiment_dir
        )


        print(
            f"\nDataset YAML:"
        )

        print(
            yaml_path
        )


        # ====================================================
        # LOAD FRESH YOLO11n-seg
        # ====================================================

        print(
            "\nLoading fresh YOLO11n-seg model..."
        )


        model = YOLO(
            MODEL_PATH
        )


        # ====================================================
        # TRAIN
        #
        # ONLY 10 REAL IMAGES ARE USED.
        # ====================================================

        print(
            "\nTraining YOLO11n-seg "
            "using ONLY the 10 real images..."
        )


        model.train(

            data=str(
                yaml_path
            ),

            epochs=EPOCHS,

            imgsz=IMAGE_SIZE,

            batch=BATCH_SIZE,

            device=DEVICE,

            project=str(
                WORK_DIR /
                "real10_training"
            ),

            name=(
                f"experiment_"
                f"{experiment_number}"
            ),

            seed=experiment_number,

            pretrained=True,

            verbose=True
        )


        # ====================================================
        # BEST WEIGHTS
        # ====================================================

        # best_weights = (
        #     WORK_DIR /
        #     "real10_training" /
        #     f"experiment_{experiment_number}" /
        #     "weights" /
        #     "best.pt"
        # )
        best_weights = (
            Path("runs/segment")
            / WORK_DIR 
            / "real10_training" 
            / f"experiment_{experiment_number}" 
            / "weights"
            / "best.pt"
        )

        if not best_weights.exists():

            raise FileNotFoundError(
                f"\nBest weights not found:\n"
                f"{best_weights}"
            )


        print(
            "\nBest model:"
        )

        print(
            best_weights
        )


        # ====================================================
        # LOAD BEST MODEL
        # ====================================================

        best_model = YOLO(
            str(best_weights)
        )


        # ====================================================
        # FINAL EVALUATION
        #
        # Use the EXISTING test folder.
        # ====================================================

        print("\n")
        print("=" * 70)

        print(
            "FINAL EVALUATION ON "
            "EXISTING INDEPENDENT TEST DATASET"
        )

        print("=" * 70)


        test_metrics = best_model.val(

            data=str(
                yaml_path
            ),

            split="test",

            imgsz=IMAGE_SIZE,

            batch=BATCH_SIZE,

            device=DEVICE,

            verbose=True
        )


        # ====================================================
        # EXTRACT METRICS
        # ====================================================

        experiment_metrics = extract_metrics(
            test_metrics
        )


        experiment_metrics[
            "experiment"
        ] = experiment_number

        experiment_metrics[
            "n_real_train"
        ] = len(selected_images)


        results.append(
            experiment_metrics
        )


        # ====================================================
        # PRINT RESULTS
        # ====================================================

        print("\n")
        print(
            f"TEST RESULTS - "
            f"EXPERIMENT {experiment_number}"
        )

        print("-" * 50)

        print(
            f"bbox mAP50  : "
            f"{experiment_metrics['bbox_mAP50']:.4f}"
        )

        print(
            f"mask mAP50  : "
            f"{experiment_metrics['mask_mAP50']:.4f}"
        )

        print(
            f"bbox recall : "
            f"{experiment_metrics['bbox_recall']:.4f}"
        )

        print(
            f"mask recall : "
            f"{experiment_metrics['mask_recall']:.4f}"
        )


    # ========================================================
    # MEAN ± STD
    # ========================================================

    print("\n")
    print("=" * 70)

    print(
        "FINAL RESULTS: 10 REAL IMAGES"
    )

    print(
        "MEAN ± STD OVER 5 EXPERIMENTS"
    )

    print("=" * 70)


    metric_names = [

        "bbox_mAP50",
        "mask_mAP50",
        "bbox_recall",
        "mask_recall"
    ]


    summary = {}


    for metric in metric_names:

        values = [
            result[metric]
            for result in results
        ]


        mean_value, std_value = (
            calculate_mean_std(values)
        )


        summary[metric] = {

            "mean": mean_value,

            "std": std_value
        }


        print(
            f"{metric:15s}: "
            f"{mean_value:.4f} ± "
            f"{std_value:.4f}"
        )


    # ========================================================
    # SAVE INDIVIDUAL RESULTS
    # ========================================================

    results_csv = (
        WORK_DIR /
        "real10_results.csv"
    )


    with open(
        results_csv,
        "w",
        newline="",
        encoding="utf-8"
    ) as f:

        writer = csv.DictWriter(

            f,

            fieldnames=[
                "experiment",
                "n_real_train",
                "bbox_mAP50",
                "mask_mAP50",
                "bbox_recall",
                "mask_recall"
            ]
        )


        writer.writeheader()

        writer.writerows(
            results
        )


    # ========================================================
    # SAVE MEAN ± STD CSV
    # ========================================================

    summary_csv = (
        WORK_DIR /
        "real10_mean_std.csv"
    )


    with open(
        summary_csv,
        "w",
        newline="",
        encoding="utf-8"
    ) as f:

        writer = csv.writer(f)


        writer.writerow([
            "metric",
            "mean",
            "std",
            "mean_plus_minus_std"
        ])


        for metric in metric_names:

            mean_value = (
                summary[metric]["mean"]
            )

            std_value = (
                summary[metric]["std"]
            )


            writer.writerow([

                metric,

                f"{mean_value:.6f}",

                f"{std_value:.6f}",

                f"{mean_value:.4f} ± "
                f"{std_value:.4f}"
            ])


    # ========================================================
    # SAVE TEXT SUMMARY
    # ========================================================

    summary_txt = (
        WORK_DIR /
        "real10_summary.txt"
    )


    with open(
        summary_txt,
        "w",
        encoding="utf-8"
    ) as f:

        f.write(
            "YOLO11n-seg: 10 Real Images Only\n"
        )

        f.write(
            "=" * 60 +
            "\n\n"
        )


        f.write(
            "Experimental protocol\n"
        )

        f.write(
            "-" * 60 +
            "\n"
        )

        f.write(
            "Training data: 10 selected real images\n"
        )

        f.write(
            "Number of experiments: 5\n"
        )

        f.write(
            "Validation data: Existing experiment/val\n"
        )

        f.write(
            "Test data: Existing experiment/test\n"
        )

        f.write(
            "Statistics: Mean ± sample standard deviation\n\n"
        )


        f.write(
            "Final Results\n"
        )

        f.write(
            "-" * 60 +
            "\n"
        )


        for metric in metric_names:

            mean_value = (
                summary[metric]["mean"]
            )

            std_value = (
                summary[metric]["std"]
            )


            f.write(

                f"{metric}: "
                f"{mean_value:.4f} ± "
                f"{std_value:.4f}\n"
            )


    # ========================================================
    # FINAL OUTPUT
    # ========================================================

    print("\n")
    print("=" * 70)
    print("ALL 5 EXPERIMENTS COMPLETED")
    print("=" * 70)


    print("\nFinal results:")


    for metric in metric_names:

        print(
            f"{metric:15s}: "
            f"{summary[metric]['mean']:.4f} ± "
            f"{summary[metric]['std']:.4f}"
        )


    print("\nSaved:")

    print(
        f"Individual results: {results_csv}"
    )

    print(
        f"Mean ± STD: {summary_csv}"
    )

    print(
        f"Text summary: {summary_txt}"
    )




## 10 real + synthetic images ranging from [0, 250, 500, 750, 1000]

In [1]:
import csv
import random
import shutil
import numpy as np
from pathlib import Path
from ultralytics import YOLO

# ============================================================
# CONFIGURATION
# ============================================================

WORK_DIR = Path("random_experiments")
N_EXPERIMENTS = 5
SYNTHETIC_SIZES = [250, 500, 750]

# Source Directories
REAL_IMAGES_DIR = Path("carrots/crack_all/images/Real")
REAL_LABELS_DIR = Path("carrots/crack_all/labels/Real")

SYNTH_IMAGES_DIR = Path("carrots/crack_all/images/CAD2Render_full")
SYNTH_LABELS_DIR = Path("carrots/crack_all/labels/CAD2Render_full")

MODEL_PATH = "yolo11n-seg.pt"

EPOCHS = 100
IMAGE_SIZE = 640
BATCH_SIZE = 16
DEVICE = 0

VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp"}


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def read_file_list(txt_file):
    """Read line-separated file list from a text file."""
    if not txt_file.exists():
        raise FileNotFoundError(f"File not found: {txt_file}")
    with open(txt_file, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]


def get_all_synthetic_images(synth_dir):
    """Scan directory and return a sorted list of all valid synthetic image filenames."""
    if not synth_dir.exists():
        raise FileNotFoundError(f"Synthetic directory does not exist: {synth_dir}")

    files = [
        f.name for f in synth_dir.iterdir()
        if f.is_file() and f.suffix.lower() in VALID_EXTENSIONS
    ]

    if not files:
        raise RuntimeError(f"No valid image files found in {synth_dir}")

    return sorted(files)


def prepare_ablation_dataset(exp_dir, real_files, selected_synth_files, synth_count):
    """
    Creates dataset directory combining 10 fixed real images 
    and a randomly sampled subset of synthetic images.
    """
    train_img_dir = exp_dir / "images" / f"train_10real_{synth_count}synth"
    train_lbl_dir = exp_dir / "labels" / f"train_10real_{synth_count}synth"

    if train_img_dir.exists():
        shutil.rmtree(train_img_dir)
    if train_lbl_dir.exists():
        shutil.rmtree(train_lbl_dir)

    train_img_dir.mkdir(parents=True, exist_ok=True)
    train_lbl_dir.mkdir(parents=True, exist_ok=True)

    # 1. Copy 10 Real Images & Labels
    for img_name in real_files:
        shutil.copy2(REAL_IMAGES_DIR / img_name, train_img_dir / img_name)
        lbl_name = f"{Path(img_name).stem}.txt"
        shutil.copy2(REAL_LABELS_DIR / lbl_name, train_lbl_dir / lbl_name)

    # 2. Copy Selected N Synthetic Images & Labels
    for img_name in selected_synth_files:
        shutil.copy2(SYNTH_IMAGES_DIR / img_name, train_img_dir / img_name)
        lbl_name = f"{Path(img_name).stem}.txt"
        synth_lbl_path = SYNTH_LABELS_DIR / lbl_name
        if not synth_lbl_path.exists():
            raise FileNotFoundError(f"Missing label file for synthetic image: {synth_lbl_path}")
        shutil.copy2(synth_lbl_path, train_lbl_dir / lbl_name)

    # 3. Create 2-Class YAML targeting images/val for validation and images/test for final metrics
    yaml_path = exp_dir / f"dataset_{synth_count}synth.yaml"
    yaml_content = f"""
path: {exp_dir.resolve()}
train: images/train_10real_{synth_count}synth
val: images/val
test: images/test

names:
  0: class0
"""
    yaml_path.write_text(yaml_content.strip(), encoding="utf-8")
    return yaml_path


def extract_metrics(metrics):
    """Extract all 4 bounding box and segmentation metrics."""
    return {
        "bbox_mAP50": float(metrics.box.map50),
        "mask_mAP50": float(metrics.seg.map50),
        "bbox_recall": float(metrics.box.r.mean()),
        "mask_recall": float(metrics.seg.r.mean())
    }


def calculate_mean_std(values):
    values = np.asarray(values, dtype=float)
    return np.mean(values), np.std(values, ddof=1)


# ============================================================
# MAIN ABLATION RUNNER
# ============================================================

def main():
    metric_names = ["bbox_mAP50", "mask_mAP50", "bbox_recall", "mask_recall"]
    all_synth_files = get_all_synthetic_images(SYNTH_IMAGES_DIR)

    for synth_count in SYNTHETIC_SIZES:
        print("\n" + "#" * 70)
        print(f" STARTING ABLATION: 10 REAL + {synth_count} SYNTHETIC IMAGES")
        print("#" * 70)

        synth_results = []

        for exp_num in range(1, N_EXPERIMENTS + 1):
            exp_dir = WORK_DIR / f"experiment_{exp_num}"

            real_files = read_file_list(exp_dir / "selected_real_images.txt")

            # Deterministically sample synthetic images per experiment seed
            rng = random.Random(exp_num)
            if len(all_synth_files) < synth_count:
                raise ValueError(
                    f"Requested {synth_count} synthetic images, but only {len(all_synth_files)} available."
                )
            selected_synth = rng.sample(all_synth_files, synth_count)

            yaml_path = prepare_ablation_dataset(exp_dir, real_files, selected_synth, synth_count)

            # Train YOLO11n-seg model
            model = YOLO(MODEL_PATH)
            model.train(
                data=str(yaml_path),
                epochs=EPOCHS,
                imgsz=IMAGE_SIZE,
                batch=BATCH_SIZE,
                device=DEVICE,
                project=str(WORK_DIR / f"synth_{synth_count}_runs"),
                name=f"experiment_{exp_num}",
                seed=exp_num,
                pretrained=True,
                verbose=False
            )

            # Locate Best Weights
            best_weights = (
                Path("runs/segment")
                / WORK_DIR
                / f"synth_{synth_count}_runs"
                / f"experiment_{exp_num}"
                / "weights"
                / "best.pt"
            )

            if not best_weights.exists():
                raise FileNotFoundError(f"Best weights not found:\n{best_weights}")

            # Evaluate Model on TEST Dataset Split
            best_model = YOLO(str(best_weights))
            test_metrics = best_model.val(
                data=str(yaml_path),
                split="test",
                imgsz=IMAGE_SIZE,
                batch=BATCH_SIZE,
                device=DEVICE,
                verbose=False
            )

            extracted = extract_metrics(test_metrics)
            extracted["experiment"] = exp_num
            extracted["synth_count"] = synth_count
            synth_results.append(extracted)

        # Print Mean ± Std on Test Set for this ablation step
        print(f"\n--- TEST RESULTS FOR {synth_count} SYNTHETIC IMAGES ---")
        summary_rows = []
        for metric in metric_names:
            vals = [r[metric] for r in synth_results]
            m, s = calculate_mean_std(vals)
            summary_rows.append([metric, f"{m:.6f}", f"{s:.6f}", f"{m:.4f} ± {s:.4f}"])
            print(f"{metric:15s}: {m:.4f} ± {s:.4f}")

        # Save Detailed CSV for this synthetic size
        results_csv = WORK_DIR / f"test_details_10real_{synth_count}synth.csv"
        with open(results_csv, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(
                f, 
                fieldnames=["experiment", "synth_count", "bbox_mAP50", "mask_mAP50", "bbox_recall", "mask_recall"]
            )
            writer.writeheader()
            writer.writerows(synth_results)

        # Save Summary CSV
        summary_csv = WORK_DIR / f"test_summary_10real_{synth_count}synth.csv"
        with open(summary_csv, "w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow(["metric", "mean", "std", "mean_plus_minus_std"])
            writer.writerows(summary_rows)



In [2]:

if __name__ == "__main__":
    main()


######################################################################
 STARTING ABLATION: 10 REAL + 250 SYNTHETIC IMAGES
######################################################################
New https://pypi.org/project/ultralytics/8.4.137 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.53 🚀 Python-3.10.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Max-Q Workstation Edition, 97248MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=random_experiments/experiment_1/dataset_250synth.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=

████████████████████████████████ 100% | 5.73/5.73 MB [00:00<00:00, 622.85MB/s]: 

Ultralytics 8.4.53 🚀 Python-3.10.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Max-Q Workstation Edition, 97248MiB)
YOLO11n-seg summary (fused): 114 layers, 2,834,763 parameters, 0 gradients, 9.6 GFLOPs


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 5316.6±414.8 MB/s, size: 393.7 KB)
val: Scanning /home/wenzhi/datasets/random_experiments/experiment_1/labels/test.cache... 36 images, 15 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 36/36 10.1Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 1.7it/s 1.8s1.5s
                   all         36        189      0.956       0.73       0.82      0.556      0.861      0.651      0.719      0.282
Speed: 3.0ms preprocess, 3.7ms inference, 0.0ms loss, 10.6ms postprocess per image
Results saved to /home/wenzhi/datasets/runs/segment/val-38
New https://pypi.org/project/ultralytics/8.4.137 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.53 🚀 Python-3.10.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Max-Q Workstation Edition, 97248MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=Fals

████████████████████████████████ 100% | 5.73/5.73 MB [00:00<00:00, 522.91MB/s]: 

...

Ultralytics 8.4.53 🚀 Python-3.10.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Max-Q Workstation Edition, 97248MiB)
YOLO11n-seg summary (fused): 114 layers, 2,834,763 parameters, 0 gradients, 9.6 GFLOPs


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 4387.0±1029.0 MB/s, size: 340.5 KB)
val: Scanning /home/wenzhi/datasets/random_experiments/experiment_4/labels/test.cache... 36 images, 15 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 36/36 10.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 1.9it/s 1.6s1.3s
                   all         36        189      0.915      0.693      0.795      0.495      0.824      0.593      0.659       0.24
Speed: 3.0ms preprocess, 1.6ms inference, 0.0ms loss, 8.6ms postprocess per image
Results saved to /home/wenzhi/datasets/runs/segment/val-51
New https://pypi.org/project/ultralytics/8.4.137 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.53 🚀 Python-3.10.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Max-Q Workstation Edition, 97248MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=Fals

████████████████████████████████ 100% | 5.73/5.73 MB [00:00<00:00, 487.20MB/s]: 

Ultralytics 8.4.53 🚀 Python-3.10.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Max-Q Workstation Edition, 97248MiB)
YOLO11n-seg summary (fused): 114 layers, 2,834,763 parameters, 0 gradients, 9.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3839.3±623.4 MB/s, size: 341.5 KB)
val: Scanning /home/wenzhi/datasets/random_experiments/experiment_5/labels/test.cache... 36 images, 15 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 36/36 9.4Mit/s 0.0s


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 2.0it/s 1.5s1.2s
                   all         36        189      0.896      0.741      0.786      0.492      0.885      0.619      0.706      0.287
Speed: 2.6ms preprocess, 1.6ms inference, 0.0ms loss, 9.2ms postprocess per image
Results saved to /home/wenzhi/datasets/runs/segment/val-52

--- TEST RESULTS FOR 750 SYNTHETIC IMAGES ---
bbox_mAP50     : 0.7806 ± 0.0211
mask_mAP50     : 0.6721 ± 0.0525
bbox_recall    : 0.7058 ± 0.0436
mask_recall    : 0.5977 ± 0.0524
